# Lab Instructions

In the lab, you're presented a task such as building a dataset, training a model, or writing a training loop, and we'll provide the code structured in such a way that you can fill in the blanks in the code using the knowledge you acquired in the chapters that precede the lab. You should be able to find appropriate snippets of code in the course content that work well in the lab with minor or no adjustments.

The blanks in the code are indicated by ellipsis (`...`) and comments (`# write your code here`).

In some cases, we'll provide you partial code to ensure the right variables are populated and any code that follows it runs accordingly.

```python
# write your code here
x = ...
```

The solution should be a single statement that replaces the ellipsis, such as:

```python
# write your code here
x = [0, 1, 2]
```

In some other cases, when there is no new variable being created, the blanks are shown like in the example below:

```python
# write your code here
...
```

Although we're showing you only a single ellipsis (`...`), you may have to write more than one line of code to complete the step, such as:

```python
# write your code here
for i, xi in enumerate(x):
    x[i] = xi * 2
```

## Installation Notes

The notebook is designed for Google Colab. A GPU is helpful for encoding the corpus, but the dataset is small enough that the lab can also run on CPU.

In [ ]:
!pip install -q -U transformers datasets

## 15.16 Lab 7: Building and Evaluating a Semantic Search System

In this lab, you will build a complete dense-retrieval experiment using **NFCorpus**, a biomedical information-retrieval dataset from the BEIR benchmark. You will write a complete the retrieval pipeline, implement the evaluation metrics, compare two document representations (text only vs title and text), and analyze retrieval errors.

### 15.16.1 Dataset

The BEIR dataset separates retrieval data into:
- a **corpus** of searchable documents;
- **queries**;
- **qrels**, which state which documents are relevant to each evaluated query.

We'll use the official **test** qrels for evaluation. So, let's load the corpus, the queries, and the query-relations (`qrels`), which are the "answers" to which entries in the corpus are the most relevant to any given query.

In [ ]:
from datasets import load_dataset

corpus = load_dataset("BeIR/nfcorpus", "corpus", split="corpus")
queries = load_dataset("BeIR/nfcorpus", "queries", split="queries")
qrels = load_dataset("BeIR/nfcorpus-qrels", split="test")
corpus, queries, qrels

Let's inspect the datasets for its columns and then take a closer look at one example from each:

In [ ]:
print("Corpus columns:", corpus.column_names)
print("Query columns:", queries.column_names)
print("Qrels columns:", qrels.column_names)

print("\nCorpus example:")
print(corpus[0])

print("\nQuery example:")
print(queries[0])

print("\nQrels example:")
print(qrels[0])

print("\nNumber of documents:", len(corpus))

A corpus row contains a unique `_id`, a `title`, and the main `text`. A query also has an ID and text. A qrels row links a query ID to a corpus ID and assigns a relevance score.

#### 15.16.1.1 Combining Title and Text

We can apply a simply function to the whole dataset to create an additional column, `title_text`, that combines the contents of both.

In [ ]:
def build_document_text(row):
    text = row["text"].strip()
    title = row["title"].strip()
    return {'title_text': f"{title}. {text}"}

corpus = corpus.map(build_document_text)
sample = corpus[0]
sample['_id'], sample['text'][:300], sample['title_text'][:300]

Later on, it will be easier to retrieve data from the corpus if we build a dictionary out of the dataset. If we set `_id` as the index, we'll have a dictionary with three entries, one for each column:

In [ ]:
corpus_dict = corpus.to_pandas().set_index('_id').to_dict()
corpus_dict.keys()

Each one of those is a dictionary as well, where the keys are the IDs:

In [ ]:
corpus_id = list(corpus_dict['text'].keys())
corpus_id[:5]

To retrieve the `text` (or `title_text` or just `title`) for any given ID, we can simply use the dictionary:

In [ ]:
corpus_dict['text']['MED-301']

#### 15.16.1.2 Organizing the Relevance Judgments

For this lab, every `qrels` row with a score greater than zero is treated as relevant. So, let's ensure all (if any) zero score gets filtered out.

In [ ]:
relevant_qrels = qrels.filter(lambda row: row['score'] > 0)
relevant_qrels

One query may have many potential references in corpus with different scores. How many unique queries do we actually have in this dataset?

In [ ]:
len(relevant_qrels.unique('query-id'))

Since there are many more entries in `qrels` than unique query IDs, let's build a dataframe that list all corpus references for each query.

In [ ]:
qrels_df = relevant_qrels.select_columns(['query-id', 'corpus-id']).to_pandas()
relevant_by_query_df = qrels_df.groupby('query-id')['corpus-id'].agg(list).reset_index()
relevant_by_query_df

Then, let's turn this dataframe into a dictionary, so we can easily retrieve all relevant references, given a query ID.

In [ ]:
relevant_dict = relevant_by_query_df.set_index('query-id')['corpus-id'].to_dict()
relevant_dict['PLAIN-1008']

The query dataset contains queries belonging to different benchmark partitions. A query should only be evaluated against the qrels for the partition being measured. Here we use the official **test qrels**, so only query IDs represented in those qrels belong in the test evaluation.

In [ ]:
relevant_ids = list(relevant_dict.keys())
test_queries = queries.filter(lambda row: row['_id'] in relevant_ids)
test_queries

Just like we did with the corpus, we can create a dictionary out of the queries, so we can retrieve their texts, given an ID:

In [ ]:
queries_dict = test_queries.select_columns(['_id', 'text']).to_pandas().set_index('_id')['text'].to_dict()
test_query_id = list(queries_dict.keys())
test_query_id[:5]

In [ ]:
print(queries_dict['PLAIN-2'])

### 15.16.2 Loading the Embedding Model

We'll use the same lightweight embedding model introduced in Chapter 15, `sentence-transformers/all-MiniLM-L6-v2`, but we'll work directly on its Transformer output so the pooling step remains visible. Also, since we're using the model as a feature extractor, it should be set to the appropriate evaluation mode (we're not training it any further).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")

repo_id = "sentence-transformers/all-MiniLM-L6-v2"
# write your code here
tokenizer = ...
model = ...
model.eval()

### 15.16.3 Mean Pooling and Text Encoding

Padding tokens must not contribute to the sentence/document representation. We therefore use the attention mask during mean pooling. For consistency with the sentence-transformer configuration, we explicitly truncate to 256 tokens. The helper functions below, `mean_pooling` and `encode_texts`, are the ones you implemented in Lab 6.

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)
    return pooled.clamp(min=1e-9)

def encode_texts(model, tokenizer, texts, batch_size=128):
    all_embeddings = []

    device = next(iter(model.parameters())).device

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.inference_mode():
            model_output = model(**inputs)
            pooled_embeddings = mean_pooling(model_output, inputs["attention_mask"])
            normalized_embeddings = F.normalize(pooled_embeddings, p=2, dim=1)

        all_embeddings.append(normalized_embeddings.cpu())

    return torch.cat(all_embeddings)

Use the appropriate function to encode a few examples from our test queries. The embeddings should have (N, 384) shape, and they should have a unit norm as well.

In [ ]:
# write your code here
sample_embeddings = ...

print(sample_embeddings.shape)
print(sample_embeddings.norm(dim=1))
assert torch.allclose(sample_embeddings.norm(dim=1), torch.ones(3), atol=1e-5)

### 15.16.4 Experiment A: Text-Only Retrieval

For the first experiment, we'll use only the main `text` from corpus to try and find the most similar IDs to any given query. So, we need to encode both the corpus and the test queries.

In [ ]:
# write your code here
corpus_embeddings_text = ...
query_embeddings = ...

print("Corpus embeddings:", corpus_embeddings_text.shape)
print("Query embeddings:", query_embeddings.shape)

#### 15.16.4.1 Exact Semantic Search

Because every embedding has unit L2 norm,

$$
\cos(\mathbf{q}, \mathbf{d})
=
\frac{\mathbf{q}\cdot\mathbf{d}}
{\|\mathbf{q}\|\|\mathbf{d}\|}
=
\mathbf{q}\cdot\mathbf{d}
$$

so a simple matrix multiplication gives cosine similarities. This means that the score matrix is simply:

```python
query_embeddings @ corpus_embeddings.T
```

So, let's compute the scores for our embeddings.

In [ ]:
# write your code here
scores_text = ...
print(scores_text.shape)

Each row corresponds to one query and each column to one document, so its shape should have as many rows as there test queries, and as many columns as there are entries in the corpus.

Out of all entries in the corpus, let's select only the top-100 matches for each query:

In [ ]:
max_k = 100
top_indices_text = torch.topk(scores_text, k=max_k, dim=-1).indices
top_indices_text, top_indices_text.shape

Then, we need to convert these indices into their corresponding query and corpus IDs. For each row, we use its corresponding query ID as key, and assign it a list of all the corresponding corpus IDs as the value.

In [ ]:
rankings_text = {}
for idx, ranking in enumerate(top_indices_text):
    docs = [corpus_id[doc_id] for doc_id in ranking.tolist()]
    query_id = test_query_id[idx]
    rankings_text[query_id] = docs

len(rankings_text)

#### 15.16.4.2 Inspecting Search Results

The dictionary contains the result of the semantic search. Let's make it easier to inspect the results by using the helper function below. For a given query ID, it uses the other dictionaries we have also created to retrieve its text and the titles and texts for the first top-K search results. 

In [ ]:
import pandas as pd

def show_results(query_id, rankings, k=5):
    print("QUERY:", queries_dict[query_id])
    print()

    relevant = relevant_dict[query_id]
    rows = []
    for rank, document_id in enumerate(rankings[query_id][:k], start=1):
        rows.append({
            "rank": rank,
            "document_id": document_id,
            "relevant": document_id in relevant,
            "title": corpus_dict["title"][document_id],
            "text": corpus_dict["text"][document_id][:300],
        })

    return pd.DataFrame(rows)

show_results(test_queries['_id'][0], rankings_text, k=5)

### 15.16.5 Retrieval Metrics

Next, we need to assess the quality of the retrieval. There are several metrics for this, such as precision@K, recall@K, and the reciprocal rank:
- the precision@K metric tells us, from the top-K retrieved documents, which fraction of it was actually relevant;
- the recall@K metric tells us what fraction of the relevant documents were retrieved in the top-K positions;
- the reciprocal rank tells us the inverse of the rank for the first relevant document identified.

In [ ]:
def precision_at_k(ranking, relevant, k):
    retrieved = ranking[:k]
    hits = sum(document_id in relevant for document_id in retrieved)
    return hits / k

def recall_at_k(ranking, relevant, k):
    if not relevant:
        return 0.0

    retrieved = ranking[:k]
    hits = sum(document_id in relevant for document_id in retrieved)
    return hits / len(relevant)

def reciprocal_rank(ranking, relevant):
    for rank, document_id in enumerate(ranking, start=1):
        if document_id in relevant:
            return 1.0 / rank

    return 0.0

Here is a quick example of these metrics. Documents "a", "c", and "x" are relevant, but the retrieved documents are, in order, "a", "b", "c", and "d".
Since the top-2 retrieved documents include one relevant ("a") and one non-relevant ("b") document, its precision@2 is 50%. Out of the three relevant documents, only "a" is in the top-2, so its recall@2 is 33%. Finally, given that the very first retrieved document ("a") is a relevant one, the reciprocal rank is the inverse of one (its rank).

In [ ]:
# Small self-checks for the metric implementations.
test_ranking = ["a", "b", "c", "d"]
test_relevant = {"a", "c", "x"}

assert precision_at_k(test_ranking, test_relevant, 2) == 0.5
assert recall_at_k(test_ranking, test_relevant, 2) == 1 / 3
assert reciprocal_rank(test_ranking, test_relevant) == 1.0
print("Metric checks passed.")

Let's use these metrics to assess the search results we got. For each query, we have its top-100 retrieved documents (`rankings_text`), and we also have the actual relevant documents for each query (`relevant_dict`). We can also try several different values of K and compare the results.

In [ ]:
def evaluate_retrieval(rankings, relevant_by_query, ks=(1, 5, 10, 100)):
    query_ids = list(rankings.keys())
    results = {}

    for k in ks:
        precisions = []
        recalls = []

        for query_id in query_ids:
            ranking = rankings[query_id]
            relevant = relevant_by_query[query_id]
            precisions.append(precision_at_k(ranking, relevant, k))
            recalls.append(recall_at_k(ranking, relevant, k))

        results[f"Precision@{k}"] = sum(precisions) / len(precisions)
        results[f"Recall@{k}"] = sum(recalls) / len(recalls)

    reciprocal_ranks = [reciprocal_rank(rankings[query_id], relevant_by_query[query_id])
                        for query_id in query_ids]
    # MRR stands for Mean Reciprocal Rank
    results["MRR@100"] = sum(reciprocal_ranks) / len(reciprocal_ranks)
    return results


metrics_text = evaluate_retrieval(rankings_text, relevant_dict)
pd.Series(metrics_text)

As `k` increases, **recall normally rises** because the system gets more opportunities to retrieve relevant documents. Precision commonly falls because lower-ranked results contain more non-relevant documents. MRR emphasizes **how early the first relevant result appears**, while Recall@100 asks how much of the full relevant set has appeared by rank 100. They answer different questions.

### 15.16.6 Experiment B: Title + Text

Now, let's re-run the experiment by changing the document representation only. Instead of using the `text` column for encoding the corpus, we'll use `title_text`. Everything else, the queries, encoder, ranking method, and relevance judgments stay fixed.

In [ ]:
# encode the corpus
# write your code here
corpus_embeddings_title_text = ...

# computes similarity scores - the queries are the same as before, so we don't need to reencode them
# write your code here
scores_title_text = ...

# retrieves the top-k search results
# write your code here
top_indices_title_text = ...

# builds the queries/results dictionary
rankings_title_text = {}
for idx, ranking in enumerate(top_indices_title_text):
    # write your code here
    docs = ...
    query_id = ...
    rankings_title_text[query_id] = docs

# evaluates the new rankings against the actual relevant queries (whcih remain the same as before)
# write your code here
metrics_title_text = ...

How do both experiments compare?

In [ ]:
comparison = pd.DataFrame({
    "Text only": metrics_text,
    "Title + text": metrics_title_text,
})

comparison

The preferred representation should be selected from the measured results, not assumed in advance. In particular:
- compare Recall@5 and Recall@10 for near-top retrieval;
- compare MRR for first-hit quality;
- compare Recall@100 for broader coverage.

A title can help because it often states the document's subject compactly. It can also hurt individual rankings because the title changes the embedding and, when input truncation occurs, consumes part of the finite token budget.

### 15.16.7 Error Analysis

The evaluation metrics give us the overall picture. But we can also learn more about our data by inspecting cases where the results are at the extremes, like perfect recall, or the complete lack thereof. To perform this assessment, we can use yet another helper function that lists the query, the recall, and the number of actual relevant entries in the corpuse associated with that query.

In [ ]:
def per_query_recall(rankings, relevant_by_query, k=10):
    rows = []
    for query_id, ranking in rankings.items():
        recall = recall_at_k(ranking, relevant_by_query[query_id], k)
        rows.append({
            "query_id": query_id,
            "query": queries_dict[query_id],
            f"Recall@{k}": recall,
            "n_relevant": len(relevant_by_query[query_id]),
        })

    return pd.DataFrame(rows)


Let's inspect a few examples for the highest and lowest recall results for a K of ten:

In [ ]:
analysis_table = per_query_recall(rankings_title_text, relevant_dict, k=10)
print("Highest Recall@10:")
high_sorted = analysis_table.sort_values(["Recall@10", "n_relevant"], ascending=[False, False])
display(high_sorted.head(3))

print("\nLowest Recall@10:")
low_sorted = analysis_table.sort_values(["Recall@10", "n_relevant"], ascending=[False,True])
display(low_sorted.tail(3))

It seems that difficult queries are those with many potentially relevant entries. Let's take the very last one, and look at its search results:

In [ ]:
# Inspect one difficult query in detail.
difficult_query_id = low_sorted.iloc[-1]["query_id"]
show_results(difficult_query_id, rankings_title_text, k=10)

For weak queries, distinguish among several possibilities:
- the embedding model did not capture the relevant relationship;
- specialized terminology favors lexical matching;
- the document was truncated;
- relevant documents discuss several subtly different aspects of the query;
- a retrieved document looks relevant to a human but is not represented in the benchmark qrels.

The last case is especially important: retrieval judgments can be incomplete, so an apparent false positive is worth inspecting rather than automatically dismissing.

### 15.16.8 Building a Reusable Search Function

We can put everything together and write a helper function to search a query over a corpus and return the top-K results of the semantic search.

In [ ]:
def search(model, tokenizer, query, corpus_id, corpus_dict, corpus_embeddings, k=5):
    # encode the query
    # write your code here
    query_embedding = ...
    
    # compute similarity metrics
    # write your code here
    scores = ...
    
    # retrieve top-K results
    # write your code here
    top = ...

    rows = []
    # for every result in the top-K
    for rank, (score, index) in enumerate(zip(top.values.tolist(), top.indices.tolist()), start=1):
        # uses the index to get the ID of the entry in the corpus
        document_id = corpus_id[index]
        # uses the corpus dictionary to get both title and text
        # for the given document ID
        rows.append({
            "rank": rank,
            "score": score,
            "document_id": document_id,
            "title": corpus_dict["title"][document_id],
            "text": corpus_dict["text"][document_id],
        })

    return pd.DataFrame(rows)

The function above requires embeddings for the corpus. Which one is better? Experiment A (text only) or B (title and text)? We need to choose a value of K and then use it to compare the resulting metrics. We'll use **Recall@10** as the primary selection metric because the intended system exposes a short ranked list while still valuing coverage.

In [ ]:
if (metrics_title_text["Recall@10"] >= metrics_text["Recall@10"]):
    best_name = "Title + text"
    best_embeddings = corpus_embeddings_title_text
    best_rankings = rankings_title_text
else:
    best_name = "Text only"
    best_embeddings = corpus_embeddings_text
    best_rankings = rankings_text

print("Selected representation:", best_name)

Once the embeddings are selected, we can come up with a query of our own and call the search function:

In [ ]:
search(model, tokenizer, "Does ginger help with nausea?", corpus_id, corpus_dict, best_embeddings, k=5)

We can also try a couple more examples. Feel free to try some queries of your own.

In [ ]:
# Two more examples.
display(search(model, tokenizer, "What dietary factors are associated with high blood pressure?", corpus_id, corpus_dict, best_embeddings, k=5))
display(search(model, tokenizer, "Can exercise reduce the risk of cardiovascular disease?", corpus_id, corpus_dict, best_embeddings, k=5))

### 15.16.9 Going Further

This lab demonstrates a very simple and straightforward way of performing semantic search. A real-world system could include several other mechanisms (which we won't be covering in this course), for example:
- chunk-level retrieval for long documents;
- a second lightweight embedding model;
- dense versus BM25 lexical retrieval;
- a hybrid lexical+dense ranking